# Error analysis — bbox-tube-temporal

Inspect FN and FP sequences from `evaluate_packaged` output. Renders
raw frames + per-tube logits so error modes can be characterized
qualitatively.

**Configure** the `VARIANT`, `SPLIT`, and `PACKAGED_SUBDIR` constants
in the next cell to switch between baseline and Track C snapshots.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from bbox_tube_temporal.aggregation_analysis import load_predictions

VARIANT = "gru_convnext_finetune"
SPLIT = "val"
PACKAGED_SUBDIR = "packaged"
MAX_ERRORS_TO_SHOW = 10
FRAMES_PER_ROW = 5

# Paths are relative to notebooks/ (the notebook's working directory).
PREDICTIONS_PATH = Path(f"../data/08_reporting/{SPLIT}/{PACKAGED_SUBDIR}/{VARIANT}/predictions.json")
SEQUENCES_ROOT = Path(f"../data/01_raw/datasets/{SPLIT}")

In [ ]:
records = load_predictions(PREDICTIONS_PATH)

false_negatives = [r for r in records if r["label"] == "smoke" and not r["is_positive"]]
false_positives = [r for r in records if r["label"] == "fp" and r["is_positive"]]
print(f"FN: {len(false_negatives)}   FP: {len(false_positives)}")

In [ ]:
def _sequence_image_paths(label: str, sequence_id: str) -> list[Path]:
    label_dir = "wildfire" if label == "smoke" else "fp"
    seq_dir = SEQUENCES_ROOT / label_dir / sequence_id / "images"
    return sorted(seq_dir.glob("*.jpg"))


def show_error(record: dict) -> None:
    paths = _sequence_image_paths(record["label"], record["sequence_id"])
    n = len(paths)
    rows = max(1, (n + FRAMES_PER_ROW - 1) // FRAMES_PER_ROW)
    fig, axes = plt.subplots(rows, FRAMES_PER_ROW, figsize=(FRAMES_PER_ROW * 2.2, rows * 2))
    axes_list = axes.flatten() if rows * FRAMES_PER_ROW > 1 else [axes]
    for i, ax in enumerate(axes_list):
        ax.axis("off")
        if i < n:
            ax.imshow(Image.open(paths[i]))
            ax.set_title(f"f{i}", fontsize=7)
    tubes_summary = ", ".join(f"{l:.2f}" for l in record["tube_logits"]) or "(no tubes)"
    score = record["score"]
    score_str = f"{score:.2f}" if score is not None else "none"
    fig.suptitle(
        f"{record['sequence_id']}  [{record['label']}]   "
        f"score={score_str}  tubes={record['num_tubes_kept']}  "
        f"logits=[{tubes_summary}]",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()

## False Negatives (smoke sequences the model missed)

In [ ]:
for r in false_negatives[:MAX_ERRORS_TO_SHOW]:
    show_error(r)

## False Positives (fp sequences the model flagged)

In [ ]:
for r in false_positives[:MAX_ERRORS_TO_SHOW]:
    show_error(r)
